In [ ]:
# %pip install --upgrade transformers --break-system-packages

In [ ]:
# %pip install auto-round --break-system-packages

In [ ]:
# %pip install git+https://github.com/sustcsonglin/flash-linear-attention.git --no-build-isolation --break-system-packages

In [ ]:
# %pip install git+https://github.com/Dao-AILab/causal-conv1d.git --no-build-isolation --break-system-packages

In [ ]:
# %pip install compressed-tensors --break-system-packages

In [2]:
import json
import os
import random
from pathlib import Path
from typing import Any

import torch
from auto_round import AutoRound, SignRoundConfig
from auto_round.utils.dataset_utils import CalibDataset
from datasets import load_dataset
from huggingface_hub import (
    HfApi,
    create_repo,
    get_token,
    hf_hub_download,
    list_repo_files,
    notebook_login,
)
from transformers import AutoModelForImageTextToText, AutoProcessor


In [3]:
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [4]:
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA Version: {torch.version.cuda}")
    gpu_count = torch.cuda.device_count()
    print(f"Total GPUs Detected: {gpu_count}\n")

    for idx in range(gpu_count):
        props = torch.cuda.get_device_properties(idx)
        print(f"--- GPU {idx} ---")
        print(f"  Name: {props.name}")
        print(f"  Compute Capability: {props.major}.{props.minor}")
        print(f"  Total VRAM: {props.total_memory / (1024**3):.2f} GB")
        print(
            f"  Allocated VRAM: {torch.cuda.memory_allocated(idx) / (1024**3):.2f} GB"
        )
        print(
            f"  Free VRAM: {(props.total_memory - torch.cuda.memory_reserved(idx)) / (1024**3):.2f} GB\n"
        )
else:
    print("No CUDA-compatible GPUs found.")

PyTorch Version: 2.14.0+cu132
CUDA Available: True
CUDA Version: 13.2
Total GPUs Detected: 1

--- GPU 0 ---
  Name: NVIDIA RTX PRO 6000 Blackwell Server Edition MIG 1g.24gb
  Compute Capability: 12.0
  Total VRAM: 23.62 GB
  Allocated VRAM: 0.00 GB
  Free VRAM: 23.62 GB



In [5]:
notebook_login()

In [6]:
MODEL_ID = "Qwen/Qwen3.5-0.8B"
HF_USER = "your_hf_username"  # Replace with your Hugging Face username
OUTPUT_BASE_DIR = "./AutoRound-0.8B"
LOCAL_PATH = "./local_model-0.8B"

In [7]:
!hf download $MODEL_ID --local-dir $LOCAL_PATH

Reconstructing (incomplete total...): |           |  0.00B /  0.00B            

Fetching 13 files:   0%|                                | 0/13 [00:00<?, ?it/s]

Fetching 13 files: 100%|██████████████████████| 13/13 [00:00<00:00, 639.80it/s]
Download complete:                                         |  0.00B            
Reconstruction complete: |                        |  0.00B /  0.00B            ✓ Downloaded
  path: /workspace/local_model-0.8B
Download complete:                                         |  0.00B            
Reconstruction complete: |                        |  0.00B /  0.00B            


In [8]:
model = AutoModelForImageTextToText.from_pretrained(
    LOCAL_PATH, dtype=torch.bfloat16, device_map="auto"
)
processor = AutoProcessor.from_pretrained(LOCAL_PATH)

tokenizer = processor.tokenizer

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

In [9]:
model

Qwen3_5ForConditionalGeneration(
  (model): Qwen3_5Model(
    (visual): Qwen3_5VisionModel(
      (patch_embed): Qwen3_5VisionPatchEmbed(
        (proj): Conv3d(3, 768, kernel_size=(2, 16, 16), stride=(2, 16, 16))
      )
      (pos_embed): Embedding(2304, 768)
      (rotary_pos_emb): Qwen3_5VisionRotaryEmbedding()
      (blocks): ModuleList(
        (0-11): 12 x Qwen3_5VisionBlock(
          (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True, bias=True)
          (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True, bias=True)
          (attn): Qwen3_5VisionAttention(
            (qkv): Linear(in_features=768, out_features=2304, bias=True)
            (proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (mlp): Qwen3_5VisionMLP(
            (linear_fc1): Linear(in_features=768, out_features=3072, bias=True)
            (linear_fc2): Linear(in_features=3072, out_features=768, bias=True)
            (act_fn): GELUTanh()
          )
      

In [19]:
def build_calibration_dataset(
    sources: list[dict[str, Any]],
    tokenizer,
    seqlen: int = 4096,
    total_nsamples: int = 512,
    output_jsonl: str = "./calib_mix_512_verified.jsonl",
    seed: int = 42,
) -> CalibDataset:
    rng = random.Random(seed)

    # 1. Resolve per-source sample quotas
    explicit_total = sum(s["num"] for s in sources if "num" in s)
    remaining_budget = max(0, total_nsamples - explicit_total)
    unassigned = [s for s in sources if "num" not in s]
    total_weight = sum(s.get("weight", 1.0) for s in unassigned) or 1.0

    quotas = []
    allocated = 0
    for idx, s in enumerate(sources):
        if "num" in s:
            q = int(s["num"])
        elif idx == len(sources) - 1:
            q = remaining_budget - allocated
        else:
            q = round(remaining_budget * (s.get("weight", 1.0) / total_weight))
            allocated += q
        quotas.append(max(1, q))

    def _passes_autoround_seqlen_check(text: str) -> tuple[bool, int]:
        """Verify exact tokenized length >= seqlen and check AutoRound's anti-repetition rule."""
        ids = tokenizer(text, add_special_tokens=False)["input_ids"]
        tok_len = len(ids)
        if tok_len < seqlen:
            return False, tok_len
        window = ids[:seqlen]
        if len(window) > 1 and seqlen > 2 and window.count(window[-1]) > seqlen // 2:
            return False, tok_len
        return True, tok_len

    def _render_row(row: Any, cfg: dict[str, Any]) -> str | None:
        filter_fn = cfg.get("filter_fn")
        if filter_fn and not filter_fn(row):
            return None

        apply_tpl = cfg.get("apply_chat_template", True)
        drop_sys_prob = cfg.get("drop_system_prob", 0.0)

        if isinstance(row, str):
            text = row.strip()
            if not text:
                return None
            return (
                tokenizer.apply_chat_template(
                    [{"role": "user", "content": text}],
                    tokenize=False,
                    add_generation_prompt=False,
                )
                if apply_tpl
                else text
            )

        if not isinstance(row, dict):
            return None

        # Case A: Multi-turn messages list (UltraData-Agent, UltraChat)
        msg_key = cfg.get("messages_field") or ("messages" if "messages" in row else None)
        if msg_key and isinstance(row.get(msg_key), list):
            msgs = list(row[msg_key])
            if msgs and msgs[0].get("role") == "system" and rng.random() < drop_sys_prob:
                msgs = msgs[1:]
            if not msgs:
                return None
            tools = row.get(cfg.get("tools_field", "tools")) or None
            if apply_tpl:
                try:
                    return tokenizer.apply_chat_template(
                        msgs, tools=tools, tokenize=False, add_generation_prompt=False
                    )
                except Exception:
                    msgs_no_sys = [m for m in msgs if m.get("role") != "system"]
                    return tokenizer.apply_chat_template(
                        msgs_no_sys or msgs, tokenize=False, add_generation_prompt=False
                    )
            return "\n\n".join(str(m.get("content", "")) for m in msgs if m.get("content"))

        # Case B: Flexible RAG / Q&A formatter
        if "rag_builder" in cfg:
            built = cfg["rag_builder"](row)
            if not built:
                return None
            user_msg, asst_msg = built
            msgs = [{"role": "user", "content": user_msg}, {"role": "assistant", "content": asst_msg}]
            return (
                tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
                if apply_tpl
                else f"{user_msg}\n\n{asst_msg}"
            )

        # Case C: Two-column instruction/output pair (OpenCodeInstruct, aya_dataset)
        fields = cfg.get("fields")
        if isinstance(fields, (list, tuple)) and len(fields) == 2 and apply_tpl:
            u_text = str(row.get(fields[0], "")).strip()
            a_text = str(row.get(fields[1], "")).strip()
            if not u_text or not a_text:
                return None
            msgs = [{"role": "user", "content": u_text}, {"role": "assistant", "content": a_text}]
            return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)

        # Case D: Single or concatenated text fields
        if isinstance(fields, (list, tuple)):
            raw_text = "\n\n".join(str(row.get(f, "")).strip() for f in fields if row.get(f))
        elif isinstance(fields, str):
            raw_text = str(row.get(fields, "")).strip()
        else:
            raw_text = next(
                (
                    str(row[c]).strip()
                    for c in ("text", "content", "prompt", "instruction")
                    if isinstance(row.get(c), str) and row[c].strip()
                ),
                "",
            )

        if not raw_text:
            return None
        return (
            tokenizer.apply_chat_template(
                [{"role": "user", "content": raw_text}],
                tokenize=False,
                add_generation_prompt=False,
            )
            if apply_tpl
            else raw_text
        )

    def _stream_raw_hf_jsonl(repo_id: str, max_items: int, stream_rng: random.Random) -> list[dict]:
        files = [f for f in list_repo_files(repo_id, repo_type="dataset") if f.endswith(".jsonl")]
        stream_rng.shuffle(files)
        rows = []
        for rel_path in files:
            if len(rows) >= max_items:
                break
            local_file = hf_hub_download(repo_id, filename=rel_path, repo_type="dataset")
            with open(local_file, "r", encoding="utf-8") as fh:
                for line in fh:
                    if len(rows) >= max_items:
                        break
                    if line.strip():
                        rows.append(json.loads(line))
        return rows

    sep = tokenizer.eos_token or "\n\n"
    sep_tokens = len(tokenizer(sep, add_special_tokens=False)["input_ids"])

    def _collect_from_source(
        cfg: dict[str, Any], target_quota: int, run_seed: int, seen_hashes: set[int]
    ) -> tuple[list[str], list[int]]:
        """Stream, pack, verify >= seqlen, and deduplicate up to `target_quota` samples."""
        src = cfg["source"]
        config_name = cfg.get("config_name")
        split = cfg.get("split", "train")
        max_scan = cfg.get("max_scan", target_quota * 50)
        local_rng = random.Random(run_seed)

        valid_samples: list[str] = []
        source_lengths: list[int] = []
        buf_parts: list[str] = []
        buf_tokens = 0
        scanned = 0

        def _try_add_verified(candidate_text: str) -> bool:
            h = hash(candidate_text)
            if h in seen_hashes:
                return False
            is_valid, tok_len = _passes_autoround_seqlen_check(candidate_text)
            if is_valid:
                seen_hashes.add(h)
                valid_samples.append(candidate_text)
                source_lengths.append(tok_len)
                return True
            return False

        def _consume_row(row: Any) -> bool:
            nonlocal buf_parts, buf_tokens, scanned
            scanned += 1
            text = _render_row(row, cfg)
            if not text:
                return False

            tok_len = len(tokenizer(text, add_special_tokens=False)["input_ids"])
            if tok_len >= seqlen:
                _try_add_verified(text)
            elif tok_len > 0:
                buf_parts.append(text)
                buf_tokens += tok_len + sep_tokens
                if buf_tokens >= seqlen:
                    packed_candidate = sep.join(buf_parts)
                    if _try_add_verified(packed_candidate):
                        buf_parts, buf_tokens = [], 0

            return len(valid_samples) >= target_quota

        if isinstance(src, list):
            for row in src:
                if _consume_row(row) or scanned >= max_scan:
                    break
        elif isinstance(src, str) and os.path.exists(src):
            p = Path(src)
            if p.suffix == ".jsonl":
                with open(p, "r", encoding="utf-8") as fh:
                    for line in fh:
                        if line.strip() and (_consume_row(json.loads(line)) or scanned >= max_scan):
                            break
            elif p.suffix == ".json":
                data = json.loads(p.read_text(encoding="utf-8"))
                for row in list(data.values()) if isinstance(data, dict) else data:
                    if _consume_row(row) or scanned >= max_scan:
                        break
        elif isinstance(src, str):
            try:
                ds = load_dataset(src, name=config_name, split=split, streaming=True).shuffle(
                    seed=run_seed, buffer_size=2000
                )
                for row in ds:
                    if _consume_row(row) or scanned >= max_scan:
                        break
            except Exception as exc:
                print(
                    f"[Fallback] Streaming '{src}' ({config_name}) failed ({type(exc).__name__}); reading raw JSONL..."
                )
                for row in _stream_raw_hf_jsonl(src, max_scan, local_rng):
                    if _consume_row(row):
                        break

        label = cfg.get("label") or (f"{src}:{config_name}" if config_name else str(src))
        min_tok = min(source_lengths) if source_lengths else 0
        print(
            f"Source '{label}': target={target_quota}, verified={len(valid_samples)}/{target_quota} "
            f"(min_tokens={min_tok}, scanned={scanned})"
        )
        return valid_samples[:target_quota], source_lengths[:target_quota]

    # 2. Primary pass across all configured sources
    all_final_texts: list[str] = []
    verified_lengths: list[int] = []
    seen_hashes: set[int] = set()
    exhausted_indices: set[int] = set()

    for idx, (cfg, quota) in enumerate(zip(sources, quotas)):
        samples, lengths = _collect_from_source(cfg, quota, seed + idx, seen_hashes)
        all_final_texts.extend(samples)
        verified_lengths.extend(lengths)
        if len(samples) < quota:
            exhausted_indices.add(idx)

    # 3. Automatic shortfall backfill from non-exhausted sources in `sources`
    shortfall = total_nsamples - len(all_final_texts)
    if shortfall > 0:
        print(f"\n[Backfill] Shortfall of {shortfall} samples detected. Topping up from available sources...")
        # Prefer sources explicitly flagged with `"topup": True`, otherwise use any non-exhausted source
        preferred_idxs = [i for i, c in enumerate(sources) if c.get("topup") and i not in exhausted_indices]
        candidate_idxs = preferred_idxs or [i for i in range(len(sources)) if i not in exhausted_indices]

        for idx in candidate_idxs:
            if shortfall <= 0:
                break
            cfg = sources[idx]
            extra_samples, extra_lengths = _collect_from_source(
                cfg, shortfall, seed + 1000 + idx, seen_hashes
            )
            all_final_texts.extend(extra_samples)
            verified_lengths.extend(extra_lengths)
            shortfall -= len(extra_samples)

    # 4. Trim if explicit quotas exceeded total_nsamples, assert length, and save
    all_final_texts = all_final_texts[:total_nsamples]
    verified_lengths = verified_lengths[:total_nsamples]
    assert all(l >= seqlen for l in verified_lengths), "Found sample shorter than seqlen!"

    combined = list(zip(all_final_texts, verified_lengths))
    rng.shuffle(combined)
    all_final_texts = [t for t, _ in combined]
    verified_lengths = [l for _, l in combined]

    with open(output_jsonl, "w", encoding="utf-8") as out_f:
        out_f.writelines(json.dumps({"text": text}, ensure_ascii=False) + "\n" for text in all_final_texts)

    print(
        f"\nVerified & saved {len(all_final_texts)}/{total_nsamples} samples -> {output_jsonl} "
        f"(min_tokens={min(verified_lengths) if verified_lengths else 0}, seqlen={seqlen})"
    )
    return CalibDataset(output_jsonl, fields="text", apply_chat_template=False, concat=True)

In [20]:
TARGET_NSAMPLES = 512
TARGET_SEQLEN = 4096

calib_sources = [
    # 1. Agentic & Tool Calling — 37.50% (192 samples)
    {
        "source": "openbmb/UltraData-SFT-Agent-2609",
        "num": 192,
        "messages_field": "messages",
        "tools_field": "tools",
        "drop_system_prob": 0.99,
        "topup": True,
    },
    # 2. Coding Assistant — 31.25% (160 samples)
    {
        "source": "nvidia/OpenCodeInstruct",
        "num": 160,
        "fields": ["input", "output"],
        "topup": True,
    },
    # 3. English General Multi-Turn Chat — 15.63% (80 samples)
    {
        "source": "HuggingFaceH4/ultrachat_200k",
        "num": 80,
        "split": "train_sft",
        "messages_field": "messages",
    },
    # 4. Multilingual Indic Chat — 15.63% (20 x 4 = 80 samples across Hindi, Tamil, Telugu, Malayalam)
    *[
        {
            "source": "CohereForAI/aya_dataset",
            "label": f"CohereForAI/aya_dataset:{lang_name}",
            "split": "train",
            "num": 20,
            "fields": ["inputs", "targets"],
            "filter_fn": lambda r, target_lang=lang_name: (
                r.get("language") == target_lang
            ),
            "max_scan": 20000,
        }
        for lang_name in ("Hindi", "Tamil", "Telugu", "Malayalam")
    ],
]

calib_dataset = build_calibration_dataset(
    sources=calib_sources,
    tokenizer=tokenizer,
    seqlen=TARGET_SEQLEN,
    total_nsamples=TARGET_NSAMPLES,
    output_jsonl="./calib_mix_512_new.jsonl",
    seed=42,
)

[Fallback] Streaming 'openbmb/UltraData-SFT-Agent-2609' (None) failed (ValueError); reading raw JSONL...
Source 'openbmb/UltraData-SFT-Agent-2609': target=192, verified=192/192 (min_tokens=6454, scanned=368)


Resolving data files:   0%|          | 0/50 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/50 [00:00<?, ?it/s]

Source 'nvidia/OpenCodeInstruct': target=160, verified=160/160 (min_tokens=4099, scanned=1354)
Source 'HuggingFaceH4/ultrachat_200k': target=80, verified=80/80 (min_tokens=4100, scanned=331)
Source 'CohereForAI/aya_dataset:Hindi': target=20, verified=5/20 (min_tokens=4164, scanned=20000)
Source 'CohereForAI/aya_dataset:Tamil': target=20, verified=20/20 (min_tokens=4102, scanned=3808)
Source 'CohereForAI/aya_dataset:Telugu': target=20, verified=20/20 (min_tokens=4108, scanned=4417)
Source 'CohereForAI/aya_dataset:Malayalam': target=20, verified=11/20 (min_tokens=4099, scanned=20000)

[Backfill] Shortfall of 24 samples detected. Topping up from available sources...
[Fallback] Streaming 'openbmb/UltraData-SFT-Agent-2609' (None) failed (ValueError); reading raw JSONL...


data/General_Agent/General_Agent_part-06(…): reconstructing file:   0%|          |  0.00B / 1.13GB            

data/General_Agent/General_Agent_part-06(…): downloading bytes:           |  0.00B            

Source 'openbmb/UltraData-SFT-Agent-2609': target=24, verified=24/24 (min_tokens=5278, scanned=24)

Verified & saved 512/512 samples -> ./calib_mix_512_new.jsonl (min_tokens=4099, seqlen=4096)


In [22]:
with open("./calib_mix_512_new.jsonl", "r", encoding="utf-8") as f:
    lines = [json.loads(line)["text"] for line in f]

lengths = [len(tokenizer(t, add_special_tokens=False)["input_ids"]) for t in lines]
double_bos = sum("<|im_start|>user\n<|im_start|>" in t for t in lines)

print(f"Total rows: {len(lines)} | Min tokens: {min(lengths)} | Max tokens: {max(lengths)} | Double-templated: {double_bos}")

Total rows: 512 | Min tokens: 4099 | Max tokens: 85451 | Double-templated: 0


In [ ]:
# Configure optimization parameters for the AutoRound / SignRound algorithm
alg_config = SignRoundConfig(
    iters=1000,                             # Optimization iterations per block; high accuracy (Production grade)
    gradient_accumulate_steps=4,            # Set to 8 / batch_size to maintain an effective global batch size of 8; smaller values risk accuracy drop
)

# Initialize the AutoRound quantization pipeline
ar = AutoRound(
    model=model,                            # Target pretrained model to be quantized
    tokenizer=tokenizer,                    # Tokenizer used to process text calibration samples
    processor=processor,                    # Processor for multimodal models (vision/audio), if present
    scheme="W4A16",                         # Quantization precision: 4-bit weights and 16-bit activations
    alg_configs=alg_config,                 # Applies the algorithm-specific optimization settings defined above
    group_size=32,                          # Number of weights sharing one scale. Cannot go below 32 (vLLM's GPTQ kernel won't run it). Only use 16 if targeting NVFP4.
    sym=True,                               # Symmetric quantization; enables faster compute and broader engine compatibility
    nsamples=512,                           # Total calibration dataset samples to maintain quality and avoid overfitting
    seqlen=4096,                            # Calibration context length; captures long-range dependency activations
    batch_size=2,                           # Batch size for calibration; keep < 8 to avoid slow execution/high memory overhead
    low_gpu_mem_usage=False,                # Keeps modules on GPU to maximize speed if sufficient VRAM is available
    enable_torch_compile=True,              # Compiles forward passes using torch.compile to accelerate calibration (JIT acceleration)
    quant_nontext_module=False,             # Leaves non-text modules (e.g., vision encoders) unquantized to protect precision (Crucial for VLM accuracy)
    dataset=calib_dataset,
    ignore_layers="mtp,visual,linear_attn", # Excludes MTP, and visual modules from quantization (Add linear-attention if needed)
    layer_config={
        "mtp": {"data_type": "bfloat16"},   # Keeps the MTP module in BF16 precision
        "mtp.fc": {
            "data_type": "bfloat16"
        },                                  # Keeps the MTP fully connected layer in BF16 precision
    },                                      # Preserves BF16 precision for the specified MTP layers
)

In [ ]:
# Quantize once and export to disk across all major loader schemas
ar.quantize_and_save(
    OUTPUT_BASE_DIR,
    # Serializes weights to 3 formats simultaneously to guarantee compatibility:
    # - 'auto_round': Reference AutoRound format (vLLM GPTQ/Marlin, Transformers / native loader)
    # - 'auto_gptq': AutoGPTQ tensor layout (vLLM GPTQ/Marlin, TGI)
    # - 'llm_compressor': Compressed-tensors schema (vLLM / Neural Magic)
    format="auto_round,auto_gptq,llm_compressor",
    # Overwrites weights in-place to avoid extra VRAM allocation
    inplace=True,
)

In [ ]:
def push_to_hub(local_dir, repo_name, token):
    """Creates repo and uploads folder to Hugging Face."""
    full_repo_id = f"{HF_USER}/{repo_name}"
    print(f"\n[Hub] Pushing {local_dir} to {full_repo_id}...")

    try:
        api = HfApi()
        create_repo(
            full_repo_id, repo_type="model", exist_ok=True, private=False, token=token
        )

        api.upload_folder(
            folder_path=local_dir,
            repo_id=full_repo_id,
            repo_type="model",
            delete_patterns=[
                "*.safetensors",
                "*.bin",
                "*.pt",
                "*.gguf",
                "*.json",
                "*.model",
                "*.onnx",
            ],
            commit_message="Update model weights and config",
        )
        print(f"[Hub] ✅ Successfully uploaded: https://huggingface.co/{full_repo_id}")
    except Exception as e:
        print(f"[Hub] ❌ Error uploading: {e}")

In [ ]:
base_name = MODEL_ID.split("/")[-1]
hf_token = get_token()

In [ ]:
if hf_token:
    push_to_hub(
        os.path.join(OUTPUT_BASE_DIR, "local_model-w4g32/auto-round-auto-gptq"),
        f"{base_name}-W4A16-AutoRound",
        hf_token,
    )
    push_to_hub(
        os.path.join(OUTPUT_BASE_DIR, "local_model-w4g32/auto-gptq"),
        f"{base_name}-W4A16-GPTQ",
        hf_token,
    )
    push_to_hub(
        os.path.join(OUTPUT_BASE_DIR, "local_model-w4g32/llm-compressor-wint-a16"),
        f"{base_name}-W4A16-LLM-Compressor",
        hf_token,
    )

else:
    print("No Hugging Face token found. Skipping upload to hub.")